# Exercise 7 — Register tests through MMIO

Run on the **PYNQ-Z2** using the bitstream already programmed by Vivado.
This notebook does not load an overlay or program the FPGA.

Configuration supplied with the snippets:

| Parameter | Value |
|---|---|
| AXI GPIO base address | `0x41200000` |
| Channel 1 | 9 output bits for commands |
| Channel 2 | 4 input bits from `q[3:0]` |
| FPGA clock | 50 MHz |
| Software reset | Not connected in the current design |

Confirm that the programmed design matches this configuration before accessing MMIO.
Start with **cells 1, 2, and 3 only**. Cell 3 performs the first AXI accesses.
An access to an unresponsive AXI peripheral can hang the board; a Python timeout
does not necessarily protect against this. If cell 3 hangs, do not run the later cells.
Check the Processing System, `M_AXI_GP0`, `FCLK_CLK0`, and AXI resets first.

The code has been formatted from the supplied snippets. The missing `select_mode()`
helper has been added in cell 4. No hardware cells have been executed here.


## Cell 1 — Prepare constants

This cell does not access the hardware.


In [ ]:
from pynq import MMIO
import time

BASE_ADDR = 0x41200000
ADDR_RANGE = 0x1000

# AXI GPIO register offsets
GPIO_DATA = 0x00
GPIO_TRI = 0x04
GPIO2_DATA = 0x08
GPIO2_TRI = 0x0C

# Command bits
SERIAL_IN = 1 << 4
MODE = 1 << 5
LOAD = 1 << 6
SHIFT = 1 << 7
RESET = 1 << 8  # Not connected to register_system/reset

DATA_MASK = 0xF
COMMAND_MASK = 0x1FF

print("AXI GPIO base:", hex(BASE_ADDR))
print("AXI GPIO range:", hex(ADDR_RANGE))
print("Ready to initialize MMIO")


## Cell 2 — Initialize MMIO

Run this cell separately. Creating the MMIO object does not guarantee that the
peripheral responds.


In [ ]:
mmio = MMIO(BASE_ADDR, ADDR_RANGE)

print("MMIO object initialized")
print("Base address:", hex(BASE_ADDR))


## Cell 3 — Check the AXI GPIO configuration

These are the first actual AXI bus accesses. For the bits used here, channel 1
must read `0` (outputs), and channel 2 must read `1` (inputs).


In [ ]:
# Run only after confirming that the correct Vivado bitstream
# is programmed and the AXI clock/reset are operational.
direction1 = mmio.read(GPIO_TRI)
direction2 = mmio.read(GPIO2_TRI)

print(f"GPIO 1 direction: 0x{direction1:08X}")
print(f"GPIO 2 direction: 0x{direction2:08X}")

assert (direction1 & 0x1FF) == 0, (
    "GPIO 1 is not configured as 9 output bits"
)
assert (direction2 & 0xF) == 0xF, (
    "GPIO 2 is not configured as 4 input bits"
)

print("AXI GPIO direction configuration: PASS")


## Cell 4 — Command helpers

Prepare data with requests inactive, raise `LOAD` or `SHIFT`, then return the
request to zero. The 10 ms pauses are intentionally long compared with the
20 ns FPGA clock period; the hardware adapter generates the single-cycle pulses.

`select_mode()` completes the helper functions required by cell 7.


In [ ]:
def write_commands(value):
    mmio.write(GPIO_DATA, value & COMMAND_MASK)
    time.sleep(0.01)


def read_register():
    return mmio.read(GPIO2_DATA) & DATA_MASK


def load_parallel(value):
    value &= DATA_MASK

    # Prepare data and select parallel output.
    write_commands(value)

    # Trigger one LOAD pulse.
    write_commands(value | LOAD)
    write_commands(value)

    return read_register()


def shift_bit(bit):
    bit = int(bool(bit))

    # Select shift register output.
    base = MODE | (SERIAL_IN if bit else 0)
    write_commands(base)
    write_commands(base | SHIFT)
    write_commands(base)

    return read_register()


def select_mode(mode):
    # Select the output while keeping LOAD and SHIFT requests inactive.
    write_commands(MODE if mode else 0)
    return read_register()


## Cell 5 — Test the parallel register

Load `1010` and check that the hardware returns it.

Expected output:

```text
Parallel register output: 1010
Parallel register test: PASS
```


In [ ]:
value = load_parallel(0b1010)

print(f"Parallel register output: {value:04b}")
assert value == 0b1010, (
    f"Parallel register test failed: {value:04b}"
)

print("Parallel register test: PASS")


## Cell 6 — Test the SIPO register

Shift in `1`, `0`, `1`, `1`. The final value must be `1011`.
Intermediate states may depend on the initial contents because the software
reset is not connected. Four shifts replace all four previous bits.


In [ ]:
sequence = [1, 0, 1, 1]

for bit in sequence:
    value = shift_bit(bit)
    print(f"Shift in {bit} -> Q = {value:04b}")

assert value == 0b1011, (
    f"Shift register test failed: {value:04b}"
)

print("Shift register test: PASS")


## Cell 7 — Test output selection and independent storage

Select and read each register without enabling either storage operation.

Expected output:

```text
Parallel register: 1010
Shift register:    1011
Register selection test: PASS
```

This notebook does not test the software reset, which is disconnected from
`register_system` in the supplied configuration.


In [ ]:
parallel_value = select_mode(0)
shift_value = select_mode(1)

print(f"Parallel register: {parallel_value:04b}")
print(f"Shift register:    {shift_value:04b}")

assert parallel_value == 0b1010
assert shift_value == 0b1011

print("Register selection test: PASS")

# Leave command requests inactive.
write_commands(0)


In [ ]:
from pathlib import Path

checks = {
    "FPGA Manager State": "/sys/class/fpga_manager/fpga0/state",
    "FPGA Bridges": "/sys/class/fpga_bridge",
    "FPGA Clock 0": "/sys/kernel/debug/clk/fclk0/clk_rate",
}

for name, path in checks.items():
    p = Path(path)
    print(f"\n{name}:")
    if p.is_file():
        try:
            print(p.read_text().strip())
        except PermissionError:
            print("Permission denied")
    elif p.is_dir():
        print([x.name for x in p.iterdir()])
    else:
        print("Not exposed by this kernel")

In [ ]:
from pynq import MMIO

SLCR_BASE = 0xF8000000
slcr = MMIO(SLCR_BASE, 0x1000)

APER_CLK_CTRL = 0x12C
value = slcr.read(APER_CLK_CTRL)

print(f"APER_CLK_CTRL: 0x{value:08X}")

print("M_AXI_GP0 clock enabled:",
      bool(value & (1 << 0)))

In [4]:
from pynq import MMIO
import time

gpio = MMIO(0x41200000, 0x1000)

GPIO_DATA = 0x00
GPIO2_DATA = 0x08

MODE_SHIFT = 0x20
SERIAL_IN = 0x10
SHIFT = 0x80

bits = [1, 0, 1, 1]

for bit in bits:
    command = MODE_SHIFT | (SERIAL_IN if bit else 0)

    # Préparer l'entrée série
    gpio.write(GPIO_DATA, command)
    time.sleep(0.01)

    # Générer une impulsion SHIFT
    gpio.write(GPIO_DATA, command | SHIFT)
    time.sleep(0.01)

    # Relâcher SHIFT
    gpio.write(GPIO_DATA, command)
    time.sleep(0.01)

    value = gpio.read(GPIO2_DATA) & 0xF
    print(f"Bit injected: {bit} -> Register: {value:04b}")

result = gpio.read(GPIO2_DATA) & 0xF

print(f"\nExpected: 0xB")
print(f"Actual:   0x{result:X}")
print("PASS" if result == 0xB else "FAIL")

Bit injected: 1 -> Register: 0001
Bit injected: 0 -> Register: 0010
Bit injected: 1 -> Register: 0101
Bit injected: 1 -> Register: 1011

Expected: 0xB
Actual:   0xB
PASS
